In [2]:
import pandas as pd 


In [3]:
df = pd.read_csv("D:/nassau_profitability_project/data/processed/nassau_featured.csv")

In [7]:
print(df.shape)
df.head()

(10194, 42)


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,country_region,city,state_province,postal_code,...,sales_per_unit,cost_per_unit,cost_ratio,markup_pct,margin_risk_flag,cost_efficiency_flag,profit_category,product_segment,revenue_contribution_pct,profit_contribution_pct
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,3.25,1.14,0.350769,185.087719,Healthy,Efficient,Low Profit,Low Performer,0.004584,0.004516
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,3.75,1.30,0.346667,188.461538,Healthy,Efficient,Low Profit,Low Performer,0.005290,0.005244
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,3.49,1.00,0.286533,249.000000,Healthy,Efficient,Medium Profit,Star Product,0.007384,0.007994
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,3.60,1.10,0.305556,227.272727,Healthy,Efficient,Medium Profit,Star Product,0.007617,0.008026
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,3.75,1.30,0.346667,188.461538,Healthy,Efficient,Medium Profit,High Sales Low Margin,0.007935,0.007866


In [8]:
# create product summary
product_summary = df.groupby(
    ["product_id", "product_name", "division"],
    as_index=False
).agg({
    "sales": "sum",
    "cost": "sum",
    "gross_profit": "sum",
    "units": "sum",
    "order_id": "nunique"
})

product_summary.rename(columns={"order_id": "total_orders"}, inplace=True)

In [9]:
#add KPI 
product_summary["gross_margin_pct"] = (product_summary["gross_profit"] / product_summary["sales"]) * 100
product_summary["profit_per_unit"] = product_summary["gross_profit"] / product_summary["units"]

In [10]:
# Add contribution %%
total_sales = product_summary["sales"].sum()
total_profit = product_summary["gross_profit"].sum()

product_summary["revenue_contribution_pct"] = (product_summary["sales"] / total_sales) * 100
product_summary["profit_contribution_pct"] = (product_summary["gross_profit"] / total_profit) * 100

In [11]:
# Add ranking 
product_summary["profit_rank"] = product_summary["gross_profit"].rank(ascending=False)
product_summary["margin_rank"] = product_summary["gross_margin_pct"].rank(ascending=False)

In [12]:
#Add segmentation again 
margin_median = product_summary["gross_margin_pct"].median()
sales_median = product_summary["sales"].median()

def product_segment(row):
    if row["sales"] >= sales_median and row["gross_margin_pct"] < margin_median:
        return "High Sales Low Margin"
    elif row["gross_profit"] >= product_summary["gross_profit"].median() and row["gross_margin_pct"] >= margin_median:
        return "Star Product"
    elif row["gross_margin_pct"] >= margin_median and row["sales"] < sales_median:
        return "High Margin Low Sales"
    else:
        return "Low Performer"

product_summary["product_segment"] = product_summary.apply(product_segment, axis=1)

In [13]:
product_summary.head()

,product_id,product_name,division,sales,cost,gross_profit,units,total_orders,gross_margin_pct,profit_per_unit,revenue_contribution_pct,profit_contribution_pct,profit_rank,margin_rank,product_segment
0,CHO-FUD-51000,Wonka Bar - Fudge Mallows,Chocolate,24890.40,8296.80,16593.60,6914,1527,66.666667,2.40,17.555200,17.758030,5.0,5.0,Star Product
1,CHO-MIL-31000,Wonka Bar - Milk Chocolate,Chocolate,26867.75,9424.38,17443.37,8267,1768,64.923077,2.11,18.949825,18.667431,3.0,7.0,Star Product
2,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Chocolate,23574.95,6755.00,16819.95,6755,1529,71.346705,2.49,16.627413,18.000263,4.0,3.0,Star Product
3,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,Chocolate,27874.80,8517.30,19357.50,7743,1704,69.444444,2.50,19.660098,20.715882,1.0,4.0,Star Product
4,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Chocolate,28485.00,9874.80,18610.20,7596,1677,65.333333,2.45,20.090472,19.916141,2.0,6.0,Star Product


** Divison Summary TAble **
- which divison is strong 
- revenue vs profit imbalance 

In [14]:
division_summary = df.groupby("division", as_index=False).agg({
    "sales": "sum",
    "cost": "sum",
    "gross_profit": "sum",
    "units": "sum"
})

In [15]:
#Add KPI
division_summary["gross_margin_pct"] = (division_summary["gross_profit"] / division_summary["sales"]) * 100
division_summary["profit_per_unit"] = division_summary["gross_profit"] / division_summary["units"]

In [16]:
# Contribution 
total_sales = division_summary["sales"].sum()
total_profit = division_summary["gross_profit"].sum()

division_summary["revenue_contribution_pct"] = (division_summary["sales"] / total_sales) * 100
division_summary["profit_contribution_pct"] = (division_summary["gross_profit"] / total_profit) * 100

In [17]:
division_summary.to_csv("D:/nassau_profitability_project/data/processed/division_summary.csv", index=False)

** Monthly Trend Table 
- For time analysis 


In [18]:
monthly_summary = df.groupby("year_month", as_index=False).agg({
    "sales": "sum",
    "cost": "sum",
    "gross_profit": "sum",
    "units": "sum"
})

In [19]:
# Add Margin 
monthly_summary["gross_margin_pct"] = (monthly_summary["gross_profit"] / monthly_summary["sales"]) * 100

In [20]:
monthly_summary.to_csv("D:/nassau_profitability_project/data/processed/monthly_summary.csv", index=False)

** factory Summary ** 


In [21]:
factory_summary = df.groupby("factory", as_index=False).agg({
    "sales": "sum",
    "cost": "sum",
    "gross_profit": "sum",
    "units": "sum"
})

In [22]:
#Add KPI 
factory_summary["gross_margin_pct"] = (factory_summary["gross_profit"] / factory_summary["sales"]) * 100

In [23]:
factory_summary.to_csv("D:/nassau_profitability_project/data/processed/factory_summary.csv", index=False)

In [24]:
#Top product
product_summary.sort_values("gross_profit", ascending=False).head(10)

,product_id,product_name,division,sales,cost,gross_profit,units,total_orders,gross_margin_pct,profit_per_unit,revenue_contribution_pct,profit_contribution_pct,profit_rank,margin_rank,product_segment
3,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,Chocolate,27874.80,8517.30,19357.50,7743,1704,69.444444,2.50,19.660098,20.715882,1.0,4.0,Star Product
4,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Chocolate,28485.00,9874.80,18610.20,7596,1677,65.333333,2.45,20.090472,19.916141,2.0,6.0,Star Product
1,CHO-MIL-31000,Wonka Bar - Milk Chocolate,Chocolate,26867.75,9424.38,17443.37,8267,1768,64.923077,2.11,18.949825,18.667431,3.0,7.0,Star Product
2,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Chocolate,23574.95,6755.00,16819.95,6755,1529,71.346705,2.49,16.627413,18.000263,4.0,3.0,Star Product
0,CHO-FUD-51000,Wonka Bar - Fudge Mallows,Chocolate,24890.40,8296.80,16593.60,6914,1527,66.666667,2.40,17.555200,17.758030,5.0,5.0,Star Product
8,OTH-LIC-15000,Lickable Wallpaper,Other,7860.00,3930.00,3930.00,393,92,50.000000,10.00,5.543658,4.205782,6.0,11.0,High Sales Low Margin
6,OTH-GUM-21000,Wonka Gum,Other,597.50,286.80,310.70,478,118,52.000000,0.65,0.421417,0.332503,7.0,10.0,High Sales Low Margin
9,SUG-EVE-47000,Everlasting Gobstopper,Sugar,130.00,26.00,104.00,13,3,80.000000,8.00,0.091689,0.111298,8.0,1.0,Star Product
7,OTH-KAZ-38000,Kazookles,Other,1205.75,1113.00,92.75,371,94,7.692308,0.25,0.850416,0.099259,9.0,15.0,High Sales Low Margin
11,SUG-HAI-55000,Hair Toffee,Sugar,76.50,17.00,59.50,17,4,77.777778,3.50,0.053955,0.063675,10.0,2.0,High Margin Low Sales


In [25]:
product_summary.sort_values("gross_margin_pct", ascending=False).head(10)

,product_id,product_name,division,sales,cost,gross_profit,units,total_orders,gross_margin_pct,profit_per_unit,revenue_contribution_pct,profit_contribution_pct,profit_rank,margin_rank,product_segment
9,SUG-EVE-47000,Everlasting Gobstopper,Sugar,130.00,26.00,104.00,13,3,80.000000,8.00,0.091689,0.111298,8.0,1.0,Star Product
11,SUG-HAI-55000,Hair Toffee,Sugar,76.50,17.00,59.50,17,4,77.777778,3.50,0.053955,0.063675,10.0,2.0,High Margin Low Sales
2,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,Chocolate,23574.95,6755.00,16819.95,6755,1529,71.346705,2.49,16.627413,18.000263,4.0,3.0,Star Product
3,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,Chocolate,27874.80,8517.30,19357.50,7743,1704,69.444444,2.50,19.660098,20.715882,1.0,4.0,Star Product
0,CHO-FUD-51000,Wonka Bar - Fudge Mallows,Chocolate,24890.40,8296.80,16593.60,6914,1527,66.666667,2.40,17.555200,17.758030,5.0,5.0,Star Product
4,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,Chocolate,28485.00,9874.80,18610.20,7596,1677,65.333333,2.45,20.090472,19.916141,2.0,6.0,Star Product
1,CHO-MIL-31000,Wonka Bar - Milk Chocolate,Chocolate,26867.75,9424.38,17443.37,8267,1768,64.923077,2.11,18.949825,18.667431,3.0,7.0,Star Product
12,SUG-LAF-25000,Laffy Taffy,Sugar,53.73,20.25,33.48,27,10,62.311558,1.24,0.037896,0.035829,12.0,8.0,High Margin Low Sales
5,OTH-FIZ-56000,Fizzy Lifting Drinks,Sugar,78.75,31.50,47.25,21,6,60.000000,2.25,0.055542,0.050566,11.0,9.0,Low Performer
6,OTH-GUM-21000,Wonka Gum,Other,597.50,286.80,310.70,478,118,52.000000,0.65,0.421417,0.332503,7.0,10.0,High Sales Low Margin


In [26]:
# Low margin high sales 
product_summary[
    (product_summary["sales"] > product_summary["sales"].median()) &
    (product_summary["gross_margin_pct"] < product_summary["gross_margin_pct"].median())
].head(10)

,product_id,product_name,division,sales,cost,gross_profit,units,total_orders,gross_margin_pct,profit_per_unit,revenue_contribution_pct,profit_contribution_pct,profit_rank,margin_rank,product_segment
7,OTH-KAZ-38000,Kazookles,Other,1205.75,1113.0,92.75,371,94,7.692308,0.25,0.850416,0.099259,9.0,15.0,High Sales Low Margin
8,OTH-LIC-15000,Lickable Wallpaper,Other,7860.00,3930.0,3930.00,393,92,50.000000,10.00,5.543658,4.205782,6.0,11.0,High Sales Low Margin


In [27]:
# check imbalance 
division_summary[["division", "revenue_contribution_pct", "profit_contribution_pct"]]

,division,revenue_contribution_pct,profit_contribution_pct
0,Chocolate,92.883008,95.057747
1,Other,6.815491,4.637543
2,Sugar,0.301502,0.304710
